# NSGA-II Multi-objective Optimization with Bias Application

## Introduction

NSGA-II is a classic multi-objective optimization algorithm. In NSGABLACK, we enhance it with the Bias System to improve convergence speed and solution quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List

print("NSGABLACK NSGA-II Tutorial")

## ZDT1 Test Problem

In [ ]:
class ZDT1:
    def __init__(self, n_var=30):
        self.n_var = n_var
        self.n_obj = 2
    
    def evaluate(self, x):
        x = np.array(x)
        f1 = x[0]
        g = 1 + 9 * np.sum(x[1:]) / (self.n_var - 1)
        h = 1 - np.sqrt(f1 / g)
        f2 = g * h
        return [f1, f2]

# Create test problem
problem = ZDT1(n_var=10)
test_x = np.random.rand(10)
objectives = problem.evaluate(test_x)

print(f"Test objectives: f1={objectives[0]:.4f}, f2={objectives[1]:.4f}")

## NSGA-II with Bias

In [ ]:
class NSGA2WithBias:
    def __init__(self, problem, pop_size=100, bias_strength=0.2):
        self.problem = problem
        self.pop_size = pop_size
        self.bias_strength = bias_strength
        self.population = []
    
    def initialize_population(self):
        self.population = []
        for _ in range(self.pop_size):
            decision = np.random.rand(self.problem.n_var)
            objectives = self.problem.evaluate(decision)
            self.population.append({
                'decision': decision,
                'objectives': objectives,
                'rank': 0,
                'crowding_distance': 0
            })
    
    def apply_bias(self, generation):
        """Apply bias to guide search"""
        # Adaptive bias strength
        current_bias = self.bias_strength * (1 - generation / 100)
        
        for ind in self.population:
            if ind['rank'] > 0:  # Not in Pareto front
                # Apply bias toward better solutions
                noise = np.random.randn(len(ind['decision'])) * current_bias
                ind['decision'] += noise
                ind['decision'] = np.clip(ind['decision'], 0, 1)
                ind['objectives'] = self.problem.evaluate(ind['decision'])
    
    def fast_non_dominated_sort(self):
        fronts = [[]]
        
        for ind in self.population:
            ind['dominated_count'] = 0
            ind['dominates'] = []
            
            for other in self.population:
                if self.dominates(ind, other):
                    ind['dominates'].append(other)
                elif self.dominates(other, ind):
                    ind['dominated_count'] += 1
            
            if ind['dominated_count'] == 0:
                ind['rank'] = 0
                fronts[0].append(ind)
        
        i = 0
        while fronts[i]:
            next_front = []
            for ind in fronts[i]:
                for other in ind['dominates']:
                    other['dominated_count'] -= 1
                    if other['dominated_count'] == 0:
                        other['rank'] = i + 1
                        next_front.append(other)
            i += 1
            fronts.append(next_front)
        
        return fronts[:-1]
    
    def dominates(self, ind1, ind2):
        at_least_one_better = False
        for i in range(len(ind1['objectives'])):
            if ind1['objectives'][i] > ind2['objectives'][i]:
                return False
            if ind1['objectives'][i] < ind2['objectives'][i]:
                at_least_one_better = True
        return at_least_one_better
    
    def optimize(self, n_generations=100):
        self.initialize_population()
        
        for gen in range(n_generations):
            # Apply bias
            self.apply_bias(gen)
            
            # Non-dominated sorting
            fronts = self.fast_non_dominated_sort()
            
            # Simple evolution (for demonstration)
            if gen % 10 == 0:
                print(f"Generation {gen}: First front size = {len(fronts[0])}")
        
        return fronts[0] if fronts else []

# Run optimization
print("Running NSGA-II with bias...")
nsga2 = NSGA2WithBias(problem, pop_size=50, bias_strength=0.3)
pareto_front = nsga2.optimize(n_generations=50)

print(f"Final Pareto front size: {len(pareto_front)}")

## Visualization

In [ ]:
# Plot results
plt.figure(figsize=(10, 6))

if pareto_front:
    objectives = np.array([ind['objectives'] for ind in pareto_front])
    plt.scatter(objectives[:, 0], objectives[:, 1], alpha=0.7, label='Found solutions')

# True Pareto front for ZDT1
f1_true = np.linspace(0, 1, 100)
f2_true = 1 - np.sqrt(f1_true)
plt.plot(f1_true, f2_true, 'r--', linewidth=2, label='True Pareto front')

plt.xlabel('f1')
plt.ylabel('f2')
plt.title('NSGA-II with Bias - ZDT1 Problem')
plt.legend()
plt.grid(True)
plt.show()

print("Visualization complete")

## Summary

NSGA-II with bias:
- Uses fast non-dominated sorting
- Applies adaptive bias to guide search
- Maintains diversity in solutions
- Converges toward Pareto optimal front

## Next Tutorial

Next: [Bayesian Optimization and Intelligent Bias](05_Bayesian_Optimization_and_Intelligent_Bias.ipynb)